# Student Test Score Prediction - Model Selection with CatBoost

## Objetivo
Encontrar la combinación óptima de variables (4-11 variables) para predecir `exam_score` usando CatBoost.

## Estrategia
- **Algoritmo:** CatBoost (manejo nativo de categóricas)
- **Combinaciones:** 1,816 combinaciones (4-11 variables)
- **Validación:** 5-fold cross-validation
- **Métrica:** Root Mean Squared Error (RMSE)

## Dataset
- 630,000 registros de entrenamiento
- 11 variables predictoras (4 numéricas + 7 categóricas)
- Target: exam_score (continuo)

## 1. Setup e Imports

In [1]:
# Imports
import pandas as pd
import numpy as np
import time
import random
import warnings
from itertools import combinations
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# Configuración
RANDOM_SEED = 42
N_FOLDS = 5
MIN_FEATURES = 4
MAX_FEATURES = 11

# Fijar seeds
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("✓ Librerías importadas correctamente")
print(f"✓ Random seed: {RANDOM_SEED}")
print(f"✓ Número de folds: {N_FOLDS}")

✓ Librerías importadas correctamente
✓ Random seed: 42
✓ Número de folds: 5


## 2. Carga de Datos

In [2]:
# Rutas
DATA_PATH = r"C:\Users\HP\OneDrive\Escritorio\David Guzzi\Github\DGKaggle\Playground Series\Season 6\Episode 1 - Predicting Student Test Scores\data\train.csv"

# Cargar datos
print("Cargando datos...")
start_time = time.time()
df = pd.read_csv(DATA_PATH)
load_time = time.time() - start_time

print(f"✓ Datos cargados en {load_time:.2f} segundos")
print(f"✓ Registros: {len(df):,}")
print(f"✓ Columnas: {len(df.columns)}")
print(f"✓ Memoria: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Mostrar primeras filas
df.head(1)

Cargando datos...
✓ Datos cargados en 1.91 segundos
✓ Registros: 630,000
✓ Columnas: 13
✓ Memoria: 257.28 MB


,id,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty,exam_score
0,0,21,female,b.sc,7.91,98.8,no,4.9,average,online videos,low,easy,78.3


In [3]:
# Info del dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630000 entries, 0 to 629999
Data columns (total 13 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                630000 non-null  int64  
 1   age               630000 non-null  int64  
 2   gender            630000 non-null  object 
 3   course            630000 non-null  object 
 4   study_hours       630000 non-null  float64
 5   class_attendance  630000 non-null  float64
 6   internet_access   630000 non-null  object 
 7   sleep_hours       630000 non-null  float64
 8   sleep_quality     630000 non-null  object 
 9   study_method      630000 non-null  object 
 10  facility_rating   630000 non-null  object 
 11  exam_difficulty   630000 non-null  object 
 12  exam_score        630000 non-null  float64
dtypes: float64(4), int64(2), object(7)
memory usage: 62.5+ MB


## 3. Identificación de Variables

In [4]:
# Variables categóricas (CatBoost las manejará nativamente)
categorical_features = [
    'gender',
    'course',
    'internet_access',
    'sleep_quality',
    'study_method',
    'facility_rating',
    'exam_difficulty'
]

# Variables numéricas
numerical_features = [
    'age',
    'study_hours',
    'class_attendance',
    'sleep_hours'
]

# Todas las features (excluir id)
feature_cols = [col for col in df.columns if col not in ['id', 'exam_score']]

# Separar X e y
X = df[feature_cols]
y = df['exam_score']

print("="*80)
print("VARIABLES DEL MODELO")
print("="*80)
print(f"Variables categóricas ({len(categorical_features)}):")
for cat in categorical_features:
    print(f"  - {cat}")
print(f"\nVariables numéricas ({len(numerical_features)}):")
for num in numerical_features:
    print(f"  - {num}")
print(f"\nTotal de predictores: {len(feature_cols)}")
print(f"Target: exam_score")
print("="*80)

VARIABLES DEL MODELO
Variables categóricas (7):
  - gender
  - course
  - internet_access
  - sleep_quality
  - study_method
  - facility_rating
  - exam_difficulty

Variables numéricas (4):
  - age
  - study_hours
  - class_attendance
  - sleep_hours

Total de predictores: 11
Target: exam_score


## 4. Definición de Funciones de Entrenamiento

In [5]:
# Hiperparámetros de CatBoost
catboost_params = {
    'loss_function': 'RMSE',
    'eval_metric': 'RMSE',
    'iterations': 100,
    'depth': 6,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'random_seed': RANDOM_SEED,
    'verbose': False,
    'thread_count': -1,  # Usar todos los cores
    'early_stopping_rounds': 10
}

def train_catboost_model(X_train, y_train, X_val, y_val, cat_features, params=None):
    """
    Entrena un modelo CatBoost con split train/val
    
    Parameters:
    -----------
    X_train, y_train : Datos de entrenamiento
    X_val, y_val : Datos de validación
    cat_features : list, nombres de columnas categóricas
    params : dict, hiperparámetros de CatBoost
    
    Returns:
    --------
    model : CatBoostRegressor entrenado
    rmse : float, RMSE en validación
    train_time : float, tiempo de entrenamiento en segundos
    """
    if params is None:
        params = catboost_params
    
    # Identificar índices de features categóricas en este subset
    cat_feature_indices = [i for i, col in enumerate(X_train.columns) if col in cat_features]
    
    start_time = time.time()
    
    # Crear modelo
    model = CatBoostRegressor(**params)
    
    # Entrenar
    model.fit(
        X_train, y_train,
        cat_features=cat_feature_indices,
        eval_set=(X_val, y_val),
        use_best_model=True,
        verbose=False
    )
    
    train_time = time.time() - start_time
    
    # Calcular RMSE en validación
    y_pred = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    
    return model, rmse, train_time


def cross_validate_model(X, y, feature_cols, cat_features, n_folds=5, params=None):
    """
    Cross-validation con K-Fold
    
    Parameters:
    -----------
    X : DataFrame completo
    y : Series target
    feature_cols : list, columnas a usar en este modelo
    cat_features : list, nombres de features categóricas (del dataset completo)
    n_folds : int, número de folds
    params : dict, parámetros de CatBoost
    
    Returns:
    --------
    dict con mean_rmse, std_rmse, fold_rmses, total_time
    """
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_SEED)
    fold_rmses = []
    fold_times = []
    
    X_subset = X[feature_cols]
    
    # Filtrar solo las categóricas que están en este subset
    cat_in_subset = [col for col in feature_cols if col in cat_features]
    
    for fold_num, (train_idx, val_idx) in enumerate(kf.split(X_subset)):
        X_train, X_val = X_subset.iloc[train_idx], X_subset.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        _, rmse, train_time = train_catboost_model(
            X_train, y_train, X_val, y_val, cat_in_subset, params
        )
        
        fold_rmses.append(rmse)
        fold_times.append(train_time)
    
    return {
        'mean_rmse': np.mean(fold_rmses),
        'std_rmse': np.std(fold_rmses),
        'fold_rmses': fold_rmses,
        'mean_time': np.mean(fold_times),
        'total_time': np.sum(fold_times)
    }

print("✓ Funciones de entrenamiento definidas")

✓ Funciones de entrenamiento definidas


## 5. Generación de Combinaciones de Variables

In [6]:
def generate_feature_combinations(feature_list, min_features=4, max_features=11):
    """
    Genera todas las combinaciones posibles de features
    
    Parameters:
    -----------
    feature_list : list, lista de nombres de features
    min_features : int, mínimo número de features por combinación
    max_features : int, máximo número de features por combinación
    
    Returns:
    --------
    list de listas, cada sublista es una combinación de features
    """
    all_combos = []
    for r in range(min_features, max_features + 1):
        all_combos.extend(combinations(feature_list, r))
    return [list(combo) for combo in all_combos]

# Generar todas las combinaciones
all_combos = generate_feature_combinations(feature_cols, MIN_FEATURES, MAX_FEATURES)

print("="*80)
print("COMBINACIONES GENERADAS")
print("="*80)
for n_features in range(MIN_FEATURES, MAX_FEATURES + 1):
    count = len([c for c in all_combos if len(c) == n_features])
    print(f"Modelos con {n_features:2d} variables: {count:4d} combinaciones")

print(f"\n{'='*80}")
print(f"TOTAL: {len(all_combos):,} combinaciones")
print(f"Total entrenamientos (con {N_FOLDS}-fold CV): {len(all_combos) * N_FOLDS:,}")
print("="*80)

# Mostrar algunas combinaciones de ejemplo
print("\nEjemplos de combinaciones:")
for i in range(min(5, len(all_combos))):
    print(f"  {i+1}. ({len(all_combos[i])} vars): {', '.join(all_combos[i])}")

COMBINACIONES GENERADAS
Modelos con  4 variables:  330 combinaciones
Modelos con  5 variables:  462 combinaciones
Modelos con  6 variables:  462 combinaciones
Modelos con  7 variables:  330 combinaciones
Modelos con  8 variables:  165 combinaciones
Modelos con  9 variables:   55 combinaciones
Modelos con 10 variables:   11 combinaciones
Modelos con 11 variables:    1 combinaciones

TOTAL: 1,816 combinaciones
Total entrenamientos (con 5-fold CV): 9,080

Ejemplos de combinaciones:
  1. (4 vars): age, gender, course, study_hours
  2. (4 vars): age, gender, course, class_attendance
  3. (4 vars): age, gender, course, internet_access
  4. (4 vars): age, gender, course, sleep_hours
  5. (4 vars): age, gender, course, sleep_quality


## 6. Benchmark - Estimación de Tiempo

In [7]:
print("="*80)
print("EJECUTANDO BENCHMARK PARA ESTIMAR TIEMPO TOTAL")
print("="*80)
print("Probando 10 combinaciones aleatorias...\n")

# Seleccionar 10 combinaciones aleatorias
sample_combos = random.sample(all_combos, min(10, len(all_combos)))
benchmark_times = []

for i, combo in enumerate(sample_combos, 1):
    print(f"  Benchmark {i}/10 - Probando {len(combo)} variables...", end=" ")
    start = time.time()
    result = cross_validate_model(X, y, combo, categorical_features, n_folds=N_FOLDS)
    elapsed = time.time() - start
    benchmark_times.append(elapsed)
    print(f"{elapsed:.2f}s (RMSE: {result['mean_rmse']:.4f})")

# Calcular estimaciones
avg_time_per_combo = np.mean(benchmark_times)
estimated_total_seconds = avg_time_per_combo * len(all_combos)
estimated_total_minutes = estimated_total_seconds / 60
estimated_total_hours = estimated_total_seconds / 3600

print(f"\n{'='*80}")
print("ESTIMACIÓN DE TIEMPO")
print("="*80)
print(f"Tiempo promedio por combinación: {avg_time_per_combo:.2f} segundos")
print(f"Total combinaciones a probar: {len(all_combos):,}")
print(f"\nTiempo estimado total:")
print(f"  - {estimated_total_minutes:.1f} minutos")
print(f"  - {estimated_total_hours:.2f} horas")
print("="*80)
print(f"\n⏰ La búsqueda completa tomará aproximadamente {estimated_total_minutes:.0f} minutos")
print("="*80)

EJECUTANDO BENCHMARK PARA ESTIMAR TIEMPO TOTAL
Probando 10 combinaciones aleatorias...

  Benchmark 1/10 - Probando 7 variables... 164.50s (RMSE: 17.6211)
  Benchmark 2/10 - Probando 4 variables... 123.87s (RMSE: 17.2638)
  Benchmark 3/10 - Probando 4 variables... 112.74s (RMSE: 18.4454)
  Benchmark 4/10 - Probando 7 variables... 190.09s (RMSE: 17.5434)
  Benchmark 5/10 - Probando 5 variables... 137.58s (RMSE: 17.1181)
  Benchmark 6/10 - Probando 5 variables... 119.25s (RMSE: 10.6636)
  Benchmark 7/10 - Probando 5 variables... 114.79s (RMSE: 18.1016)
  Benchmark 8/10 - Probando 4 variables... 123.53s (RMSE: 10.8325)
  Benchmark 9/10 - Probando 7 variables... 154.88s (RMSE: 16.7889)
  Benchmark 10/10 - Probando 4 variables... 141.09s (RMSE: 10.8234)

ESTIMACIÓN DE TIEMPO
Tiempo promedio por combinación: 138.23 segundos
Total combinaciones a probar: 1,816

Tiempo estimado total:
  - 4183.8 minutos
  - 69.73 horas

⏰ La búsqueda completa tomará aproximadamente 4184 minutos


## 7. Búsqueda Exhaustiva - Probar Todas las Combinaciones

In [ ]:
def parallel_combination_search(X, y, cat_features, all_combos, n_folds=5):
    """
    Ejecuta búsqueda secuencial con tracking detallado
    CatBoost ya paraleliza internamente, búsqueda secuencial es suficiente
    
    Parameters:
    -----------
    X : DataFrame con features
    y : Series con target
    cat_features : list de nombres de features categóricas
    all_combos : list de combinaciones a probar
    n_folds : int, número de folds para CV
    
    Returns:
    --------
    DataFrame con resultados de todas las combinaciones
    """
    results = []
    n_total = len(all_combos)
    
    print(f"{'='*80}")
    print(f"INICIANDO BÚSQUEDA DE MODELO ÓPTIMO CON CATBOOST")
    print(f"{'='*80}")
    print(f"Total combinaciones a probar: {n_total:,}")
    print(f"Folds por combinación: {n_folds}")
    print(f"Total entrenamientos: {n_total * n_folds:,}")
    print(f"Variables categóricas: {len(cat_features)}")
    print(f"{'='*80}\n")
    
    start_time = time.time()
    
    # Usar tqdm para progress bar
    for idx, combo in enumerate(tqdm(all_combos, desc="Probando combinaciones")):
        result = cross_validate_model(X, y, combo, cat_features, n_folds)
        
        result_dict = {
            'combination_id': idx + 1,
            'n_features': len(combo),
            'features': ','.join(combo),
            'mean_rmse': result['mean_rmse'],
            'std_rmse': result['std_rmse'],
            'fold_rmses': str(result['fold_rmses']),
            'cv_time_seconds': result['total_time']
        }
        results.append(result_dict)
        
        # Print detallado cada 50 combinaciones
        if (idx + 1) % 50 == 0:
            elapsed = time.time() - start_time
            avg_time = elapsed / (idx + 1)
            remaining = (n_total - idx - 1) * avg_time
            
            print(f"\n{'='*80}")
            print(f"PROGRESO: {idx+1}/{n_total} ({100*(idx+1)/n_total:.1f}%)")
            print(f"{'='*80}")
            print(f"Tiempo transcurrido: {elapsed/60:.1f} min")
            print(f"Tiempo estimado restante: {remaining/60:.1f} min")
            print(f"Mejor RMSE hasta ahora: {min(r['mean_rmse'] for r in results):.4f}")
            print(f"Tiempo promedio por combo: {avg_time:.2f}s")
            print(f"{'='*80}\n")
    
    total_time = time.time() - start_time
    print(f"\n{'='*80}")
    print(f"BÚSQUEDA COMPLETADA")
    print(f"{'='*80}")
    print(f"Tiempo total: {total_time/60:.1f} minutos ({total_time/3600:.2f} horas)")
    print(f"Promedio por combinación: {total_time/n_total:.2f} segundos")
    print(f"{'='*80}\n")
    
    return pd.DataFrame(results)

# EJECUTAR BÚSQUEDA
results_df = parallel_combination_search(X, y, categorical_features, all_combos, n_folds=N_FOLDS)

## 8. Análisis de Resultados

In [ ]:
# Ordenar por RMSE
results_df = results_df.sort_values('mean_rmse').reset_index(drop=True)

print("="*100)
print("TOP 20 MODELOS CON MENOR RMSE")
print("="*100)
print(results_df[['combination_id', 'n_features', 'mean_rmse', 'std_rmse', 'features']].head(20).to_string(index=False))
print("="*100)

In [ ]:
# Estadísticas por número de variables
print("\n" + "="*80)
print("ESTADÍSTICAS DE RMSE POR NÚMERO DE VARIABLES")
print("="*80)
stats_by_nfeatures = results_df.groupby('n_features')['mean_rmse'].agg([
    ('count', 'count'),
    ('mean', 'mean'),
    ('std', 'std'),
    ('min', 'min'),
    ('max', 'max')
])
print(stats_by_nfeatures)
print("="*80)

In [ ]:
# Mejor modelo
best = results_df.iloc[0]

print("\n" + "="*100)
print("🏆 MEJOR MODELO ENCONTRADO")
print("="*100)
print(f"Combination ID: {best['combination_id']}")
print(f"Número de variables: {best['n_features']}")
print(f"RMSE promedio (5-fold CV): {best['mean_rmse']:.6f}")
print(f"Desviación estándar: {best['std_rmse']:.6f}")
print(f"\nVariables seleccionadas:")
for i, var in enumerate(best['features'].split(','), 1):
    print(f"  {i}. {var}")
print("="*100)

## 9. Visualizaciones

In [ ]:
# Configurar estilo
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Crear figura con subplots
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Análisis de Resultados - Búsqueda de Modelo Óptimo', fontsize=16, fontweight='bold')

# 1. Distribución de RMSE
ax1 = axes[0, 0]
ax1.hist(results_df['mean_rmse'], bins=50, edgecolor='black', alpha=0.7)
ax1.axvline(best['mean_rmse'], color='red', linestyle='--', linewidth=2, label=f'Mejor: {best["mean_rmse"]:.4f}')
ax1.set_xlabel('Mean RMSE', fontsize=12)
ax1.set_ylabel('Frecuencia', fontsize=12)
ax1.set_title('Distribución de RMSE - Todas las Combinaciones', fontsize=13, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. RMSE por número de variables (boxplot)
ax2 = axes[0, 1]
data_by_nfeatures = [results_df[results_df['n_features']==n]['mean_rmse'].values 
                     for n in range(MIN_FEATURES, MAX_FEATURES + 1)]
bp = ax2.boxplot(data_by_nfeatures, labels=range(MIN_FEATURES, MAX_FEATURES + 1),
                 patch_artist=True, showmeans=True)
for patch in bp['boxes']:
    patch.set_facecolor('lightblue')
ax2.set_xlabel('Número de Variables', fontsize=12)
ax2.set_ylabel('RMSE', fontsize=12)
ax2.set_title('RMSE por Número de Variables', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)

# 3. RMSE promedio por número de variables (line plot)
ax3 = axes[1, 0]
mean_by_n = results_df.groupby('n_features')['mean_rmse'].mean()
ax3.plot(mean_by_n.index, mean_by_n.values, marker='o', linewidth=2, markersize=8)
ax3.set_xlabel('Número de Variables', fontsize=12)
ax3.set_ylabel('RMSE Promedio', fontsize=12)
ax3.set_title('RMSE Promedio vs Número de Variables', fontsize=13, fontweight='bold')
ax3.grid(True, alpha=0.3)
ax3.set_xticks(range(MIN_FEATURES, MAX_FEATURES + 1))

# 4. Top 10 mejores modelos
ax4 = axes[1, 1]
top_10 = results_df.head(10)
bars = ax4.barh(range(10), top_10['mean_rmse'].values, color='skyblue', edgecolor='black')
bars[0].set_color('gold')  # Mejor modelo en dorado
ax4.set_yticks(range(10))
ax4.set_yticklabels([f"#{i+1} ({row['n_features']} vars)" 
                     for i, (_, row) in enumerate(top_10.iterrows())])
ax4.set_xlabel('RMSE', fontsize=12)
ax4.set_title('Top 10 Mejores Modelos', fontsize=13, fontweight='bold')
ax4.invert_yaxis()
ax4.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('rmse_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Visualizaciones guardadas en 'rmse_analysis.png'")

## 10. Entrenar Modelo Final con Mejores Variables

In [ ]:
# Obtener mejores features
best_features = best['features'].split(',')
cat_in_best = [col for col in best_features if col in categorical_features]
cat_indices = [i for i, col in enumerate(best_features) if col in categorical_features]

print("="*80)
print("ENTRENANDO MODELO FINAL CON MEJORES VARIABLES")
print("="*80)
print(f"Variables seleccionadas: {len(best_features)}")
print(f"Variables categóricas: {len(cat_in_best)}")
print("="*80)

# Entrenar modelo final en todo el dataset
final_params = catboost_params.copy()
final_params['verbose'] = True  # Mostrar progreso

final_model = CatBoostRegressor(**final_params)
final_model.fit(
    X[best_features], y,
    cat_features=cat_indices
)

print("\n✓ Modelo final entrenado")

In [ ]:
# Feature importance
importance_df = pd.DataFrame({
    'feature': best_features,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\n" + "="*60)
print("FEATURE IMPORTANCE DEL MEJOR MODELO")
print("="*60)
print(importance_df.to_string(index=False))
print("="*60)

In [ ]:
# Visualizar feature importance
plt.figure(figsize=(10, 6))
plt.barh(importance_df['feature'], importance_df['importance'], color='steelblue', edgecolor='black')
plt.xlabel('Importance', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.title('Feature Importance - Mejor Modelo', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Gráfico guardado en 'feature_importance.png'")

## 11. Exportar Resultados

In [ ]:
# 1. Exportar todos los resultados
results_df.to_csv('results_all_combinations.csv', index=False)
print("✓ Resultados completos guardados en 'results_all_combinations.csv'")

# 2. Exportar top 10
results_df.head(10).to_csv('top_10_models.csv', index=False)
print("✓ Top 10 modelos guardados en 'top_10_models.csv'")

# 3. Guardar lista de mejores features
with open('best_model_features.txt', 'w') as f:
    f.write(f"Best Model - RMSE: {best['mean_rmse']:.6f}\n")
    f.write(f"Number of features: {best['n_features']}\n\n")
    f.write("Selected features:\n")
    for var in best_features:
        f.write(f"- {var}\n")
print("✓ Mejores features guardadas en 'best_model_features.txt'")

# 4. Guardar feature importance
importance_df.to_csv('feature_importance.csv', index=False)
print("✓ Feature importance guardada en 'feature_importance.csv'")

# 5. Guardar modelo final
final_model.save_model('best_catboost_model.cbm')
print("✓ Modelo final guardado en 'best_catboost_model.cbm'")

print("\n" + "="*80)
print("TODOS LOS RESULTADOS HAN SIDO EXPORTADOS")
print("="*80)

## 12. Resumen Final

In [ ]:
print("\n" + "="*100)
print("RESUMEN FINAL - MODEL SELECTION")
print("="*100)
print(f"\n📊 DATASET:")
print(f"  - Registros: {len(df):,}")
print(f"  - Variables totales: {len(feature_cols)}")
print(f"  - Categóricas: {len(categorical_features)}")
print(f"  - Numéricas: {len(numerical_features)}")

print(f"\n🔍 BÚSQUEDA:")
print(f"  - Combinaciones probadas: {len(results_df):,}")
print(f"  - Rango de variables: {MIN_FEATURES}-{MAX_FEATURES}")
print(f"  - Cross-validation: {N_FOLDS} folds")
print(f"  - Total modelos entrenados: {len(results_df) * N_FOLDS:,}")

print(f"\n🏆 MEJOR MODELO:")
print(f"  - Combination ID: {best['combination_id']}")
print(f"  - Número de variables: {best['n_features']}")
print(f"  - RMSE promedio: {best['mean_rmse']:.6f}")
print(f"  - Desviación estándar: {best['std_rmse']:.6f}")

print(f"\n📈 DISTRIBUCIÓN DE RMSE:")
print(f"  - RMSE mínimo: {results_df['mean_rmse'].min():.6f}")
print(f"  - RMSE máximo: {results_df['mean_rmse'].max():.6f}")
print(f"  - RMSE promedio: {results_df['mean_rmse'].mean():.6f}")
print(f"  - RMSE mediana: {results_df['mean_rmse'].median():.6f}")

print(f"\n🔝 VARIABLES MÁS IMPORTANTES:")
for i, (_, row) in enumerate(importance_df.head(5).iterrows(), 1):
    print(f"  {i}. {row['feature']}: {row['importance']:.4f}")

print(f"\n💾 ARCHIVOS GENERADOS:")
print(f"  - results_all_combinations.csv")
print(f"  - top_10_models.csv")
print(f"  - best_model_features.txt")
print(f"  - feature_importance.csv")
print(f"  - best_catboost_model.cbm")
print(f"  - rmse_analysis.png")
print(f"  - feature_importance.png")

print("\n" + "="*100)
print("✅ MODEL SELECTION COMPLETADO EXITOSAMENTE")
print("="*100)

print(f"\n📝 Próximos pasos:")
print(f"  1. Revisar las variables seleccionadas en best_model_features.txt")
print(f"  2. Analizar feature importance para entender patrones")
print(f"  3. Considerar ingeniería de features en próxima sesión")
print(f"  4. Optimizar hiperparámetros del modelo final")
print(f"  5. Generar predicciones para test.csv")